In [1]:
FND_ROOT = "/workspace/rumor-detection"

In [2]:
import openai
import time
import pandas as pd

from typing import List, Dict

In [3]:
OPENAI_API_KEY = ""

In [4]:
client = openai.OpenAI(
    api_key=OPENAI_API_KEY
)  # O usa OPENAI_API_KEY como variable de entorno


In [11]:
def construir_prompt(respuesta: str) -> str:
    return f"""
Evalúa el siguiente texto según los siguientes criterios, asignando una nota del 1 al 5 con una justificación breve por cada uno.
Considera que el texto es un mensaje de Twitter y tiene un máximo de caracteres.

1. Calidad del contenido
2. Corrección gramatical
3. Pertinencia del mensaje
4. Adecuación del lenguaje

Texto:
---
{respuesta}
"""

In [12]:
def evaluar_respuestas(respuestas: List[str]) -> List[Dict]:
    resultados = []
    for respuesta in respuestas:
        prompt = construir_prompt(respuesta)
        try:
            response = client.chat.completions.create(
                model="gpt-4o",
                messages=[
                    {
                        "role": "system",
                        "content": "Eres un experto en evaluación lingüística y de redacción de textos en español.",
                    },
                    {"role": "user", "content": prompt},
                ],
            )
            contenido = response.choices[0].message.content
            resultados.append({"respuesta": respuesta, "evaluacion": contenido})
            time.sleep(1.5)
        except Exception as e:
            resultados.append({"respuesta": respuesta, "error": str(e)})
    return resultados


In [7]:
big_big_df = pd.read_pickle(f"{FND_ROOT}/notebooks/bigbigdf.pkl")

In [8]:
big_big_df.head()

,index,is_root,label,tree_id,text,n_palabras,n_oraciones,prom_oracion,token_length_mean,token_length_median,...,proportion_bullet_points,contains,duplicate_line_chr_fraction,duplicate_paragraph_chr_fraction,duplicate_ngram_chr_fraction,top_ngram_chr_fraction,oov_ratio,entropy,perplexity,per_word_perplexity
1299150874774831106,0,True,false,1299150874774831106,Esto es el colmo:\nVíctor Manoli dueño de empr...,31.0,1.0,31.0,4.838710,2.0,...,"value=0.0 passed=True threshold=(None, 0.8)",{'lorem ipsum': value=0.0 passed=True threshol...,"value=0.0 passed=True threshold=(None, 0.2)","value=0.0 passed=True threshold=(None, 0.2)","{'5': value=0.0 passed=True threshold=(None, 0...","{'2': value=0.0 passed=True threshold=(None, 0...","value=0.17 passed=True threshold=(None, 0.2)",1.384309,3.992066,0.110891
1299156214069317636,1,False,n/a,1299150874774831106,@hernan_sr @Eneatipo7 Uy k raro 🙄,6.0,1.0,6.0,4.666667,3.0,...,"value=0.0 passed=True threshold=(None, 0.8)",{'lorem ipsum': value=0.0 passed=True threshol...,"value=0.0 passed=True threshold=(None, 0.2)","value=0.0 passed=True threshold=(None, 0.2)","{'5': value=0.0 passed=True threshold=(None, 0...","{'2': value=0.0 passed=True threshold=(None, 0...","value=0.33 passed=False threshold=(None, 0.2)",0.001126,1.001127,0.166854
1299156316770963458,2,False,n/a,1299150874774831106,@hernan_sr No tienen verguenza. Que asco me da...,21.0,3.0,7.0,6.190476,4.0,...,"value=0.0 passed=True threshold=(None, 0.8)",{'lorem ipsum': value=0.0 passed=True threshol...,"value=0.0 passed=True threshold=(None, 0.2)","value=0.0 passed=True threshold=(None, 0.2)","{'5': value=0.0 passed=True threshold=(None, 0...","{'2': value=0.0 passed=True threshold=(None, 0...","value=0.23 passed=False threshold=(None, 0.2)",0.370748,1.448818,0.055724
1299158535805304832,3,False,n/a,1299150874774831106,"@hernan_sr Anciano decrépito ,cuanto te que de...",38.0,1.0,38.0,3.763158,3.0,...,"value=0.0 passed=True threshold=(None, 0.8)",{'lorem ipsum': value=0.0 passed=True threshol...,"value=0.0 passed=True threshold=(None, 0.2)","value=0.0 passed=True threshold=(None, 0.2)","{'5': value=0.0 passed=True threshold=(None, 0...","{'2': value=0.0 passed=True threshold=(None, 0...","value=0.1 passed=True threshold=(None, 0.2)",1.229832,3.420653,0.083431
1299159444501213184,4,False,n/a,1299150874774831106,@hernan_sr Está la pura cagada en el gobierno ...,14.0,2.0,7.0,4.714286,4.0,...,"value=0.0 passed=True threshold=(None, 0.8)",{'lorem ipsum': value=0.0 passed=True threshol...,"value=0.0 passed=True threshold=(None, 0.2)","value=0.0 passed=True threshold=(None, 0.2)","{'5': value=0.0 passed=True threshold=(None, 0...","{'2': value=0.0 passed=True threshold=(None, 0...","value=0.06 passed=True threshold=(None, 0.2)",0.803055,2.232351,0.139522


In [9]:
big_big_df.loc["1299150874774831106"].text

'Esto es el colmo:\nVíctor Manoli dueño de empresas de camiones, es el Intendente de la Araucanía 🤦\u200d♂️\n¡hablenme de conflicto de INTERÉS!\n#ParoDeFascistas https://t.co/X8TAOvwrWw'

In [13]:
evaluar_respuestas([big_big_df.loc["1299150874774831106"].text])

[{'respuesta': 'Esto es el colmo:\nVíctor Manoli dueño de empresas de camiones, es el Intendente de la Araucanía 🤦\u200d♂️\n¡hablenme de conflicto de INTERÉS!\n#ParoDeFascistas https://t.co/X8TAOvwrWw',
  'evaluacion': '1. Calidad del contenido: 4/5  \nEl mensaje es claro y directo, transmite una crítica sobre un posible conflicto de interés respecto a un funcionario público. Se utiliza un recurso visual, el emoji, y un hashtag que refuerzan la crítica, provocando una posible llamada a la acción o al debate entre los lectores. Sin embargo, la argumentación sería más sólida con más contexto o datos de fondo.\n\n2. Corrección gramatical: 4/5  \nLa estructura gramatical del mensaje es en gran medida correcta. Sin embargo, hay un pequeño error: "hablenme" debería llevar tilde y escribirse "háblenme" para indicar el imperativo plural.\n\n3. Pertinencia del mensaje: 4/5  \nEl tema del mensaje es pertinente y de interés público, sobre todo en un contexto de debate político o social, que suele